# BÁO CÁO THỰC HÀNH LAB 4: TỔNG HỢP TIẾNG NÓI (SPEECH SYNTHESIS)
**Môn học:** CSE457 - Xử lý âm thanh và tiếng nói  
**Học viên / Mã SV:** Hồ Đức Minh - 2351260669  
**Mục tiêu:** Thực hiện từ Text Normalization, Concatenative Synthesis, Unit Selection, Trajectory HMM đến Neural TTS.

In [1]:
import sys
from pathlib import Path
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
import librosa

# Import module bài lab
import Lab04
import hmm_tts_backend
import neural_tts_backend
print("Modules loaded successfully!")

## 1. Yêu cầu 4.1: Text Normalization & Phân tích Inventory
Tạo inventory 16 từ miền hẹp (2 candidate/unit), chuẩn hóa Fs=16kHz, RMS=0.08 và phân tích độ dài duration.

In [2]:
Lab04.run_inventory_analysis()
from IPython.display import Image
Image("figures/fig1_inventory_stats.png")

## 2. Yêu cầu 4.2 & 4.3: Baseline Concatenation, Unit Selection & Cross-Fade Smoothing
- Thử nghiệm nối trực tiếp (raw concat 0ms)
- Viterbi Unit Selection tối ưu hóa ma trận Join Cost trên log-magnitude spectrum
- Làm trơn biên nối bằng linear cross-fade (0, 5, 15, 30 ms).

In [3]:
case_phrase = "Mời khách hàng số 23 đến quầy 5."
tokens, paths, units_audio, raw_audio, boundaries = Lab04.run_baseline_concatenation(case_phrase)
opt_cands, xfade_outs = Lab04.run_unit_selection_and_smoothing(tokens, raw_audio, boundaries)
Image("figures/fig2_boundary_spectrogram.png")

In [4]:
Image("figures/fig3_unit_selection_cost.png")

## 3. Yêu cầu 4.4: Prosody & Thí nghiệm Source-Filter
Đo F0 (Pitch), thực hiện Pitch-shift ($\pm 2$ semitones) và Time-stretch ($0.8\times, 1.2\times$).

In [5]:
Lab04.run_prosody_experiments("moi_01.wav")

## 4. Yêu cầu 4.5: HMM-based Statistical Parametric Synthesis
Khảo sát chuỗi state sequence, đường F0 contour và các quỹ đạo Formant F1, F2.

In [6]:
y_hmm, tracks = Lab04.run_hmm_synthesis(case_phrase)
Image("figures/fig4_hmm_parameter_tracks.png")

## 5. Yêu cầu 4.6 & 4.7: Neural TTS Offline & So sánh 3 phương pháp
Inference 3 câu test qua mô hình neural 2-stage (PostNet + Griffin-Lim / Phase vocoder), đo latency và tổng hợp đánh giá.

In [7]:
test_phrases = [
    "Mời khách hàng số 23 đến quầy 5.",
    "Mời khách hàng số 104 đến quầy 2.",
    "Mời khách hàng số 89 đến quầy 7."
]
neural_stats = Lab04.run_neural_tts(test_phrases)
y_neural, _, _ = neural_tts_backend.synthesize(case_phrase)
Lab04.run_comparison_and_figure5(raw_audio, xfade_outs[15], y_hmm, y_neural)
Image("figures/fig5_output_comparison.png")